# Data mining demo (API required)

This demo combines **abstract triage, positive extraction, and negative reconstruction**. Read four abstracts, inspect the model requests, compare predictions with human labels, and mine positive and negative synthesis records from an article/SI pair.

Select the **Python (MOFinder demos)** kernel. For another environment, install `python -m pip install -e ".[triage,mining,notebook]"` from the repository root.

All three live switches default to `False`: `RUN_TRIAGE`, `RUN_POSITIVE_EXTRACTION`, and `RUN_NEGATIVE_RECONSTRUCTION`. Enable only the workflows you want to run. Live calls require an OpenAI API key and access to the configured models, send the selected texts to OpenAI, and incur API charges. A hidden prompt accepts your key if `OPENAI_API_KEY` is not already set. Do not paste a key into notebook code.

## Inputs

Paths below are relative to `Demo/03_api_data_mining/`.

| Input | Location | Contents |
| --- | --- | --- |
| Abstract metadata | `inputs/triage_metadata.csv` | Twelve real papers; the demonstration selects the first four in file order. |
| Human reference | `inputs/triage_ground_truth.csv` | Reference labels used locally for comparison, never sent to GPT. |
| Extraction example | `inputs/extraction/` | An illustrative article/SI pair, DOI inventory, and checksums; these PDFs are not copies of the published article. |
| Replacement PDFs | `literature_input/` | Three DOI-named article/SI templates; replace them with real documents before enabling local-paper extraction. |
| Configuration | `configs/` | One shared triage configuration and the extraction and reconstruction settings. |

Generated outputs go under `results/examples/03_api_data_mining/`. Preview mode displays inputs and validates the extraction inputs without making model requests. Positive extraction must run before negative reconstruction because it supplies the extracted CSV and synthesis JSON files.

This notebook starts with cleared outputs. Run the preview cells to inspect inputs and settings; model calls remain disabled until selected. Section 1.4 displays predictions and human-label comparisons after a live triage run. Generated records are saved in the run folders described below.

Implementation: [demonstration runner](mof_api_data_mining_demo.py), [abstract triage](../../src/mofinder/literature/triage.py), and the [source-to-code guide](../../docs/source_to_code.md).


## 1. Abstract triage

**Y** means eligible experimental MOF synthesis by traditional solution-phase methods; **N** means outside that scope, following [the screening criteria](../../prompts/abstract_triage.txt).

## 1.1. Import and read the abstracts

Each record includes the title, journal, keywords, and complete abstract. The same four records are used in the requests below. No PDFs are needed.


In [ ]:
from pathlib import Path
from html import escape
import getpass
import json
import os

from IPython.display import HTML, display
from mofinder.config import load_triage_config
from mofinder.display import display_path, display_paths
from mofinder.literature.triage import (
    build_screening_request, doi_key, read_table, save_csv, screen, validate_inputs,
)

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "Demo/03_api_data_mining/inputs/triage_metadata.csv").is_file()
)
DEMO = ROOT / "Demo/03_api_data_mining"
CONFIG_FILE = DEMO / "configs" / "triage.json"
def read_triage_inputs(config_file):
    current_config = load_triage_config(config_file)
    if len(current_config["models"]) != 1 or current_config["n_rounds"] != 1:
        raise ValueError("This demonstration supports one model and one round. See docs/triage.md for other settings.")
    current_validated = validate_inputs(
        current_config["input_file"], current_config["ground_truth_file"],
        sheet=current_config["input_sheet"], benchmark_only=current_config["benchmark_only"],
        max_papers=current_config["max_papers"],
    )
    current_prompt = current_config["prompt_file"].read_text(encoding="utf-8")
    return current_config, current_validated, current_prompt


config, validated, prompt = read_triage_inputs(CONFIG_FILE)
papers = validated["papers"]
model_config = config["models"][0]


def show_table(rows):
    """Display complete, escaped values without truncating long text."""
    columns = list(rows[0]) if rows else []
    header = "".join(f"<th>{escape(name)}</th>" for name in columns)
    body = "".join(
        "<tr>" + "".join(
            f'<td style="text-align:left;white-space:pre-wrap">{escape(str(row[name]))}</td>'
            for name in columns
        ) + "</tr>" for row in rows
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))


print(f"Loaded {len(papers)} of {validated['metadata_rows']} records from {display_path(config['input_file'])}")
for number, paper in enumerate(papers, 1):
    display(HTML(
        f"<h3>{number}. {escape(paper['title'])}</h3>"
        f"<p><b>DOI:</b> {escape(paper['DOI'])}<br>"
        f"<b>Journal:</b> {escape(paper['source'])}<br>"
        f"<b>Author keywords:</b> {escape(paper['author_keywords'])}<br>"
        f"<b>Keywords Plus:</b> {escape(paper['keywords_plus'])}</p>"
        f'<p style="white-space:pre-wrap"><b>Abstract:</b> {escape(paper["abstract"])}</p>'
    ))


## 1.2. Inspect exactly what the classifier will receive

The shared request builder below is also used by the screening function. Each request contains the eligibility instructions and one paper's metadata and abstract, with no human labels. Expand a paper to see its full prompt and request settings. GPT is asked to return a single `Y` or `N`.


In [ ]:
preview_requests = [
    build_screening_request(
        paper, model_config, prompt=prompt,
        max_output_tokens=config["max_output_tokens"],
    )
    for paper in papers
]
print(f"Ready: {len(preview_requests)} papers, {model_config['model']}, one round. No requests sent yet.")
for number, (paper, request) in enumerate(zip(papers, preview_requests), 1):
    settings = {key: value for key, value in request.items() if key != "input"}
    opened = " open" if number == 1 else ""
    display(HTML(
        f"<details{opened}><summary>Request {number}: {escape(paper['title'])}</summary>"
        f"<p><b>Request settings</b></p><pre>{escape(json.dumps(settings, indent=2))}</pre>"
        '<p><b>User message</b></p><pre style="white-space:pre-wrap">'
        + escape(request["input"][0]["content"]) + "</pre></details>"
    ))


## 1.3. Send the abstracts to the LLM

The default in [configs/triage.json](configs/triage.json) is `gpt-5` with `reasoning_effort: "high"`. The request preview in step 1.2 shows both settings. Edit that configuration to choose another supported model or reasoning effort, then rerun steps 1.1 and 1.2.

The token cap is 25,000 for reasoning and the final answer combined; the requested answer remains a single `Y` or `N`. This cap is an allowance, not the expected answer length.

Change `RUN_TRIAGE` to `True` and run this cell. The key is read from `OPENAI_API_KEY`, or requested through a hidden input prompt. Do not paste a key into notebook code.

If you edit the input files, prompt, or configuration, rerun steps 1.1 and 1.2 before sending requests. A preflight check stops the run if the saved files no longer match the preview.

This makes one classification request per abstract. Rerunning this cell with `RUN_TRIAGE = True` starts a new run and makes new billable requests. The SDK can retry transient failures. A missing key stops execution before a run is created; an API error is recorded without inventing a prediction.


In [ ]:
RUN_TRIAGE = False  # Set True to obtain real predictions for the four abstracts.
screening_run = None

if RUN_TRIAGE:
    current_config, current_validated, current_prompt = read_triage_inputs(CONFIG_FILE)
    current_requests = [
        build_screening_request(
            paper, current_config["models"][0], prompt=current_prompt,
            max_output_tokens=current_config["max_output_tokens"],
        )
        for paper in current_validated["papers"]
    ]
    if (current_config != config or current_validated != validated
            or current_prompt != prompt or current_requests != preview_requests):
        raise ValueError("Triage inputs or settings changed. Rerun steps 1.1 and 1.2 before enabling requests.")
    api_key = os.environ.get("OPENAI_API_KEY", "").strip()
    if not api_key:
        api_key = getpass.getpass("OpenAI API key (hidden): ").strip()
    try:
        if not api_key:
            raise ValueError("An API key is required. No requests were sent.")
        screening_run = await screen(CONFIG_FILE, api_key=api_key)
    finally:
        api_key = None
else:
    print("Preview only: no GPT predictions. Set RUN_TRIAGE = True in this cell, then run it and the results cell below.")


## 1.4. Inspect predictions and compare with the human labels

The table joins the real API records to the reference labels saved with that run by DOI. Later edits to the input reference do not change this comparison; papers without a reference label remain unscored for agreement. It shows raw answers and errors as well as the accepted `Y`/`N` prediction. Invalid responses, failed requests, and requests skipped after a configuration error stay **unscored**. A human label never fills in a missing GPT prediction.

Agreement on four examples is a demonstration, not a reliable estimate of model accuracy. Screening from an abstract also does not verify that the full paper contains a usable synthesis procedure.


In [ ]:
comparison = []
if screening_run is None:
    print("No API run to display. Complete step 1.3 with RUN_TRIAGE = True first.")
else:
    # Compare with the reference captured for this run, even if inputs were edited later.
    _, saved_reference, _ = read_table(screening_run.output_dir / "reference_used.csv")
    reference_by_doi = {doi_key(row["DOI"]): row for row in saved_reference}
    records_by_doi = {doi_key(row["DOI"]): row for row in screening_run.rows}
    for paper in screening_run.papers:
        key = doi_key(paper["DOI"])
        record = records_by_doi.get(key, {})
        prediction = record.get("Agent_YN", "") if record.get("Status") == "ok" else ""
        reference = reference_by_doi.get(key, {}).get("Consensus GT", "").strip().upper()
        comparison.append({
            "DOI": paper["DOI"],
            "Title": paper["title"],
            "GPT prediction": prediction or "Unscored",
            "Human reference": reference or "Unavailable",
            "Agreement": ("Yes" if prediction == reference else "No") if prediction and reference else "Unscored",
            "Status": record.get("Status", "not_requested"),
            "Raw answer": record.get("Raw answer", ""),
            "Error": record.get("Error", "") or ("No request recorded; see run_manifest.json." if not record else ""),
        })
    show_table(comparison)
    scored = [row for row in comparison if row["Status"] == "ok"]
    referenced = [row for row in scored if row["Agreement"] != "Unscored"]
    matched = sum(row["Agreement"] == "Yes" for row in referenced)
    print(f"Valid GPT answers: {len(scored)}/{len(comparison)} abstracts.")
    if referenced:
        print(f"Agreement with human labels: {matched}/{len(referenced)} valid answers with reference labels.")
    else:
        print("No valid predictions with reference labels to compare. Check the status and error columns.")

    comparison_file = screening_run.output_dir / "demo_comparison.csv"
    save_csv(comparison, comparison_file)
    print("Comparison:", display_path(comparison_file))
    print("Run records:", display_path(screening_run.output_dir))


Each live run creates a separate folder inside `MOFinder/results/examples/03_api_data_mining/triage/`, containing:

- `demo_comparison.csv`: the displayed predictions and reference-label comparison.
- `predictions.csv` and `responses.jsonl`: parsed answers, raw answer text, errors, model and response IDs, token usage, and timing.
- `run_manifest.json`: input and prompt fingerprints, settings, and completion/coverage counts.
- `reference_used.csv` and `reference_not_screened.csv`: reference labels and which papers fell outside this small run.

To try more of the bundled papers, change `max_papers` in [configs/triage.json](configs/triage.json) (up to 12), then rerun from step 1.1. The example is configured for one model and one round. For multiple models, repeated rounds, new collections, or analysis of saved runs, use [the full triage guide](../../docs/triage.md) and [screening code](../../src/mofinder/literature/triage.py).

Implementation: [screening code](../../src/mofinder/literature/triage.py). API reference: [OpenAI Responses API](https://developers.openai.com/api/reference/python/resources/responses/methods/create).


## 2. Positive extraction

The included PDFs under `inputs/extraction/` describe MOF-303 and CAU-23 using illustrative synthesis conditions. They are demonstration material, not the published article. The [document manifest](inputs/extraction/manifest.json) records their checksums.

First validate the selected documents below. Keep `EXTRACTION_CONFIG_DIR = DEMO / "configs"` for the sample pair. To use three to five real papers, replace the PDFs in `literature_input/main/` and `literature_input/si/`, update `literature_input/inventory.csv`, and select `DEMO / "configs/local_papers"`. Validation identifies unfilled templates and live extraction refuses to send them.


In [ ]:
import runpy

# Load this demonstration's runner by its exact path, avoiding another run_demo module in the kernel.
api_runner = runpy.run_path(str(DEMO / "mof_api_data_mining_demo.py"))
EXTRACTION_CONFIG_DIR = DEMO / "configs"
# For replacement PDFs: EXTRACTION_CONFIG_DIR = DEMO / "configs/local_papers"

extraction_validation = api_runner["validate_positive"](EXTRACTION_CONFIG_DIR)
print(json.dumps(display_paths(extraction_validation), indent=2, default=str))


Set `RUN_POSITIVE_EXTRACTION` to `True` to extract synthesis records. Positive extraction uses the system/user prompts and model configured in `configs/positive_extraction.json` (or `configs/local_papers/positive_extraction.json`). It writes a CSV and the synthesis JSON store needed by negative reconstruction.


In [ ]:
RUN_POSITIVE_EXTRACTION = False

if RUN_POSITIVE_EXTRACTION:
    positive_result = api_runner["run_positive"](EXTRACTION_CONFIG_DIR)
    print(json.dumps(display_paths(positive_result), indent=2, default=str))
else:
    print("Positive extraction is disabled. Enable RUN_POSITIVE_EXTRACTION to request extraction.")


## 3. Negative reconstruction

Run positive extraction first. Negative reconstruction selects documents marked `YES` for trial or failure evidence, creates modification plans, and enumerates their saved options locally. Depending on the extracted evidence, it may produce no eligible documents or no enumerated records. These reconstructed conditions are not individually verified experimental failures.

Set `RUN_NEGATIVE_RECONSTRUCTION` to `True` only after the positive CSV and synthesis JSON files are available. The settings are defined by the negative reconstruction configuration in the same `EXTRACTION_CONFIG_DIR`.


In [ ]:
RUN_NEGATIVE_RECONSTRUCTION = False

if RUN_NEGATIVE_RECONSTRUCTION:
    negative_result = api_runner["run_negative"](live=True, config_dir=EXTRACTION_CONFIG_DIR)
    print(json.dumps(display_paths(negative_result), indent=2, default=str))
else:
    print("Negative reconstruction is disabled. Complete positive extraction before enabling RUN_NEGATIVE_RECONSTRUCTION.")


## Saved files and command-line use

Default outputs are under `results/examples/03_api_data_mining/`:

- `triage/`: separate timestamped runs with predictions, comparisons, and request records.
- `extraction/`: matched documents, positive extraction CSV/JSON, and negative reconstruction outputs.
- `local_papers/`: the corresponding outputs when using the replacement-PDF configuration.

Triage creates a new run on each live invocation. Positive extraction and negative reconstruction use saved artifacts and may skip completed documents; use fresh output paths in the configuration for a new experiment.

The same workflows are available through [mof_api_data_mining_demo.py](mof_api_data_mining_demo.py). Run `python Demo/03_api_data_mining/mof_api_data_mining_demo.py validate` from the repository root for local input checks, or see [README.md](README.md) for live commands and replacement-input instructions.
